Working on automating the addition of DRN & RIV to SFR via MVR if connect_DRN, connect_RIV options are set to True in SFR_settings for Sim in WS_Mdl.imod.prep.py.

# General imports

In [4]:
import geopandas as gpd
import numpy as np
import pandas as pd
from WS_Mdl.core.defaults import CRS
from WS_Mdl.core.mdl import Mdl_N
from WS_Mdl.imod.mf6.bin import to_DF
from WS_Mdl.core.style import sprint, Sep, bold

from pathlib import Path

from WS_Mdl.core.runtime import timed_Exe
from WS_Mdl.imod.prj import PrSimP
from WS_Mdl.imod.sfr.prsimp import connect_SFR_lines_to_MF6, create_SFR_lines

# Prerequisites
Functions that need to have run before we add DRN & RIV to SFR

In [5]:
from WS_Mdl.imod.prep import Sim, SFR_settings

In [6]:
MdlN = 'NBr58'
M = Mdl_N(MdlN)
MdlN_SFR_GPkg = 'NBr54'

In [7]:
Pa_SFR_GPkg = M.Pa.In / f'SFR/{MdlN_SFR_GPkg}/WBD_1ry_SW_NW_cleaned_{MdlN_SFR_GPkg}.gpkg'
Pa_SW_Cond_A = M.Pa.WS / r"models\NBr\In\RIV\RIV_Cond_DETAILWATERGANGEN_NBr1.IDF"
Pa_SW_Cond_B = M.Pa.WS / r"models\NBr\In\RIV\RIV_Cond_DRN_NBr1.IDF"
Pa_SFR_OBS_In = M.Pa.In / 'OBS/SFR/NBr40/NBr40_SFR_OBS_Pnt.csv'
Pa_Shp_catchment = M.Pa.WS / r'models\NBr\PoP\common\Pgn\Chaamse_beek\catchment_chaamsebeek_ulvenhout.shp'

In [10]:
SFR_Cfg = SFR_settings( Pa_Cond_A = Pa_SW_Cond_A,
                        Pa_Cond_B = Pa_SW_Cond_B,
                        Pa_Gpkg = Pa_SFR_GPkg,
                        Pa_OBS_In = Pa_SFR_OBS_In,
                        connect_Pkgs=('DRN', 'RIV'),
                        Pa_Shp_connect_Pkgs = Pa_Shp_catchment
)

## Sim

In [11]:
M.verbose = True

In [12]:
sprint(Sep)

# Create Mdl_N instance and enchance with params needed in following functions.
sprint(' -- Loading MdlN parameters.', end='', verbose_in=True, verbose_out=M.verbose)
M = Mdl_N(MdlN)
M.verbose = True
sprint('🟢', verbose_in=True, verbose_out=M.verbose)

# %% Load PRJ & regrid it to Mdl Aa
sprint(' -- imod PrSimP from PRJ file.', verbose_in=True, verbose_out=M.verbose)
M.Sim_MF6, M.MSW_Mdl = timed_Exe(PrSimP, M)

# %% Create SFR Lines
sprint(' -- SFRmaker - Creating SFR lines.', verbose_in=True, verbose_out=M.verbose)
M.lines = timed_Exe(create_SFR_lines, Pa_GPkg=SFR_Cfg.Pa_Gpkg, verbose=M.verbose)

# %% Connect SFR Lines to MF6 (writes files and connects them to NAM)
sprint(' -- SFRmaker - Connecting SFR lines to MF6.', verbose_in=True, verbose_out=M.verbose)
M.Pa_SFR_OBS_In = Path(SFR_Cfg.Pa_OBS_In)
M.Pa_Cond_A = Path(SFR_Cfg.Pa_Cond_A)
M.Pa_Cond_B = Path(SFR_Cfg.Pa_Cond_B) if SFR_Cfg.Pa_Cond_B is None else Path(SFR_Cfg.Pa_Cond_B)
M.DF_reach = timed_Exe(
    connect_SFR_lines_to_MF6,
    M,
    debug_sfr=True,
)


----------------------------------------------------------------------------------------------------

 -- Loading MdlN parameters.🟢
 -- imod PrSimP from PRJ file.
  - Loading NBr58.prj ... 🟢 [123.0s]
  - Regridding PRJ ...🟢 [0.9 s]
  - Loading MF6 Simulation ... 🟢 [76.6s]
Created corrected mete_grid.inp: G:\models\NBr\In\CAP\mete_grid\NBr1\temp\mete_grid.inp
  - Loading MSW Simulation ... 🟢 [3.4s]
  - Clipping models ...🟢 [17.2 s]
  - Loading models into memory ...🟢 [15.8 s]
  - Creating mask ...🟢 [2.0 s]
  - Cleaning up MF6 packages ...🟢 [1.7 s]
  - Cleaning up MSW packages ...🟢 [0.0 s]
  - Coupling ... 🟢 [0.0s]
  - Writing model files ... 🟢 [29.3s]
 -- SFRmaker - Creating SFR lines.
 -- Create SFR lines.
  - Ensure slope.
      +----------+------------+------------+
|       ID |   Elv_UStr |   Elv_DStr |
+==========+============+============+
|   312    |  312       |  312       |
+----------+------------+------------+
|  6323.73 |    9.2364  |    8.87544 |
+----------+------------+-

# Args

In [13]:
# MdlN = 'NBr58'
# M = Mdl_N(MdlN)
Pkgs = ['DRN', 'RIV']
Pa_Shp = M.Pa.WS / r'models\NBr\PoP\common\Pgn\Chaamse_beek\catchment_chaamsebeek_ulvenhout.shp'
if isinstance(Pkgs, str):
    Pkgs = tuple(Pkgs)
# def Pkg_to_SFR_via_MVR(M: Mdl_N, Pkg, Pa_Shp):  # 666 needs a lot of cleanup and streamlinging

# Function specific imports

In [14]:
import re
import WS_Mdl.core.df  # noqa: F401
from scipy.spatial.distance import cdist
from shapely.geometry import LineString

# Load shapefile

In [15]:
if Pa_Shp is not None:
    GDF_Shp = gpd.read_file(Pa_Shp)
    GDF_Shp.crs = CRS
    print(f'Loaded shapefile with {len(GDF_Shp)} features')
    print(f'Bounds: {GDF_Shp.bounds}')

Loaded shapefile with 1 features
Bounds:           minx         miny         maxx         maxy
0  113434.7999  387938.1329  124760.5671  395868.0663


# Load Ins as DFs and combine them into one GDF

In [16]:
d_DF = {}
for Pkg in Pkgs:
    l_Pa = [i for i in M.Pa.Sim_In.rglob(f'*{Pkg.lower()}*.bin')]
    
    for Pa in l_Pa:
        PkgN = re.search(r'^.*?\d+', Pa.parent.name).group()

        DF = to_DF(Pa, Pkg=Pkg) # Load
        DF = DF.loc[~DF['i'].isin([1, M.N_R]) & ~DF['j'].isin([1, M.N_C]), ['k', 'i', 'j']] # Remove boundary cells
        DF = DF.ws.Calc_XY(M.Xmin, M.Ymax, M.cellsize)
        DF['Pkg1'] = PkgN
        DF['Pvd_ID'] = DF.index + 1  # 1-based index
        
        d_DF[PkgN] = DF

DF_all = pd.concat(d_DF.values(), ignore_index=True)
GDF_all = gpd.GeoDataFrame(DF_all, geometry=gpd.points_from_xy(DF_all.x, DF_all.y), crs=CRS)
GDF = gpd.sjoin(GDF_all, GDF_Shp, how='inner', predicate='within')

In [17]:
d_DF

{'drn-1':        k    i    j         x         y   Pkg1  Pvd_ID
 114    1    2    2  113137.5  396162.5  drn-1     115
 115    1    2    3  113162.5  396162.5  drn-1     116
 116    1    2    4  113187.5  396162.5  drn-1     117
 117    1    2    5  113212.5  396162.5  drn-1     118
 118    1    2  143  116662.5  396162.5  drn-1     119
 ...   ..  ...  ...       ...       ...    ...     ...
 22664  1  343  352  121887.5  387637.5  drn-1   22665
 22665  1  343  353  121912.5  387637.5  drn-1   22666
 22666  1  343  354  121937.5  387637.5  drn-1   22667
 22667  1  343  355  121962.5  387637.5  drn-1   22668
 22668  1  343  356  121987.5  387637.5  drn-1   22669
 
 [22490 rows x 7 columns],
 'drn-2':         k    i    j         x         y   Pkg1  Pvd_ID
 481     1    2    2  113137.5  396162.5  drn-2     482
 482     1    2    3  113162.5  396162.5  drn-2     483
 483     1    2    4  113187.5  396162.5  drn-2     484
 484     1    2    5  113212.5  396162.5  drn-2     485
 485     1   

In [18]:
Pa.parent.name

'riv-5riv'

# Calc DF_reach XY

In [28]:
M.DF_reach[[ 'i', 'j']] = M.DF_reach[[ 'i', 'j']] + 1

In [29]:
DF_reach = M.DF_reach[['rno', 'i', 'j']].ws.Calc_XY(M.Xmin, M.Ymax, M.cellsize)

# Calculate distances and find closest reach for each DRN point

In [38]:
coords = GDF[['x', 'y']].values
reach_coords = DF_reach[['x', 'y']].values
distances = cdist(coords, reach_coords, metric='euclidean')
min_indices = np.argmin(distances, axis=1)

In [39]:
# Add matched reach data to DRN DataFrame
matched_reach_data = DF_reach.iloc[min_indices].reset_index(drop=True)
DF_match = GDF.drop(columns='geometry').copy()
DF_match['Rcv_ID'] = matched_reach_data['rno'].values
DF_match['distance_to_match'] = distances[np.arange(len(coords)), min_indices]

sprint(f'Combined {len(GDF):,} points ({Pkgs}) from {len(d_DF)} DataFrames', indent=2, verbose_in=True, verbose_out=M.verbose)
sprint(f'Matched to {DF_match["Rcv_ID"].nunique()} unique reaches', indent=2, verbose_in=True, verbose_out=M.verbose)
sprint(f'Mean distance: {DF_match["distance_to_match"].mean():.0f}m', indent=2, verbose_in=True, verbose_out=M.verbose)
sprint(f'Perfect matches (same cell): {(DF_match["distance_to_match"] == 0).sum():,}', indent=2, verbose_in=True, verbose_out=M.verbose)

    Combined 105,843 points (['DRN', 'RIV']) from 6 DataFrames
    Matched to 3661 unique reaches
    Mean distance: 238m
    Perfect matches (same cell): 4,782


# Plot connections

In [54]:
t = 0
if M.verbose:
    DF_match_plot = DF_match.merge(DF_reach, left_on='Rcv_ID', right_on='rno', suffixes=('', '_r'))
    GDF_match = gpd.GeoDataFrame(DF_match,
                                geometry=[LineString([(r.x, r.y), (r.x_r, r.y_r)]) for _, r in DF_match_plot.iterrows()],
                                crs='EPSG:28992') # Create GDF with LineString Geom
    
    GDF_match = GDF_match.drop(columns=['index_right', 'CATCHMENT_', 'SUM_OPPERV', 'x', 'y']).rename(columns={'distance_to_match': 'distance', 'Pkg1': 'Pkg'}) # Trim extra Cols + rename
    
    for Pkg in GDF_match['Pkg'].unique():
        GDF = GDF_match.loc[GDF_match['Pkg'] == Pkg]
        print(f'Package: {Pkg} - Shape: {GDF.shape}')
        t += GDF.shape[0]
        Pa = M.Pa.PoP / f"In/MVR/{MdlN}/{Pkg}_to_SFR_{MdlN}.gpkg"
        Pa.parent.mkdir(parents=True, exist_ok=True)

        # GDF.to_file(Pa)

Package: drn-1 - Shape: (8935, 8)
Package: drn-2 - Shape: (79653, 8)
Package: riv-1 - Shape: (17255, 8)


# Prepare DF_w for MVR

In [ ]:
DF_match['Pkd2'] = 'sfr'
DF_w = DF_match[['Pkg1', 'Pvd_ID', 'Pkd2', 'Rcv_ID']]
DF_w['MVR_TYPE'] = 'FACTOR'
DF_w['value'] = 1

### Write MVR file

In [ ]:
Pa_MVR = M.Pa.Sim_In / f'{M.MdlN}.MVR6'

In [ ]:
with open(Pa_MVR, 'w') as f:
    f.write(f"""BEGIN OPTIONS
END OPTIONS

BEGIN DIMENSIONS
  MAXMVR {DF_w.shape[0]}
  MAXPACKAGES {len(DF_match['Pkg1'].unique()) + 1}
END DIMENSIONS

BEGIN PACKAGES
  {'\n  '.join([k for k in DF_match['Pkg1'].unique()])}
  sfr
END PACKAGES

BEGIN PERIOD 1
""")
    f.write(DF_w.ws.to_MF_block(indent=1))
    f.write('END PERIOD')

## Insert MVR line to NAM

In [ ]:
with open(M.Pa.NAM_Mdl, 'r') as f1:
    l_Lns_NAM = f1.readlines()

l_Lns_NAM.insert(-1, f'  MVR6 imported_model/{Pa_MVR.name} MVR\n')

with open(M.Pa.NAM_Mdl, 'w') as f2:
    f2.writelines(l_Lns_NAM)


## Add MOVER option to SFR

In [ ]:
with open(M.Pa.SFR, 'r') as f1:
    l_Lns_SFR = f1.readlines()

l_Lns_SFR.insert(3, '  MOVER\n')

with open(M.Pa.SFR, 'w') as f2:
    f2.writelines(l_Lns_SFR)

## Add MOVER option to DRN files

In [ ]:
for i in DF_match['Pkg1'].unique():
    Pa = list(M.Pa.Sim_In.rglob(f'{i}*.{i[:3].lower()}'))[0]
    with open((Pa), 'r') as f1:
        l_Lns = f1.readlines()

    l_Lns.insert(3, '  MOVER\n')

    with open((Pa), 'w') as f2:
        f2.writelines(l_Lns)